# LLM Disagreement on Central-Bank Stance — Prediction · Boundary Mechanism · Measurement Sensitivity

Self-contained, top-to-bottom. Upload the six `turn_predictions_<model>.csv` files (or point
`DATA_DIR` at them) and **Run all**. Needs a **GPU** runtime.

**Claim.** Text predicts *when* zero-shot LLM stance labels become unstable; that instability is a
structured, **boundary-localized** property of central-bank language, and it is economically
consequential for the spillover VAR.

**Models:** DeepSeek V3 · Gemini 2.5 Flash · GPT-4o Mini · Llama 3.3 · Mistral Large · Qwen 2.5 72B

**Outputs**
1. **Baseline + deep ladder** — dict / TF-IDF / frozen DeBERTa / fine-tuned DeBERTa, predicting
   disagreement (`split` AUC and `score_std3` Spearman).
2. **Boundary mechanism** — a stancedness classifier trained on *unanimous* turns shows the
   *split* turns are decision-boundary cases.
3. **Lexical diagnostics** — which language families (hedging, conditionality, mixed-risk, policy
   action, inflation/labour condition) put a turn on the boundary (TF-IDF coefficients + sentence
   leave-one-out, no API).
4. **Meeting-level risk** — `meeting_disagreement_for_var.csv` for the spillover step.

*Multi-head imitation and the swap decomposition are intentionally dropped — they live in the lab
notebook `multihead_turns_colab.ipynb`.* All headline numbers use a **grouped split by meeting**.

In [ ]:
!pip install -q transformers accelerate sentencepiece scikit-learn shap scipy matplotlib seaborn

In [ ]:
import os, re, random
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DebertaV2Model
from scipy.stats import entropy, spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATA_DIR = "/content/" if os.path.exists("/content") else "../output/stance/"

MODELS = {
    "deepseekv3":      "DeepSeek V3",
    "gemini25flash":   "Gemini 2.5 Flash",
    "gpt-4o":          "GPT-4o Mini",
    "llama33":         "Llama 3.3",
    "mistrallarge_or": "Mistral Large",
    "qwen25_72b":      "Qwen 2.5 72B",
}
loaded = list(MODELS.keys())

LABEL2IDX   = {"dovish":0,"mostly dovish":1,"neutral":2,"mostly hawkish":3,"hawkish":4}
LABEL_ORDER = list(LABEL2IDX.keys())
STANCED     = {"dovish","mostly dovish","mostly hawkish","hawkish"}
SCORE3      = {"dovish":-1,"mostly dovish":-1,"neutral":0,"mostly hawkish":1,"hawkish":1}
DIR2        = {"dovish":"dove","mostly dovish":"dove","mostly hawkish":"hawk","hawkish":"hawk"}
MODEL_NAME  = "microsoft/deberta-v3-base"

def path_for(key):
    return f"{DATA_DIR}turn_predictions_{key}.csv"

In [ ]:
# ---- load + health check ----
raw, rows = {}, []
for key in MODELS:
    p = path_for(key); assert os.path.exists(p), f"MISSING: {p}"
    df = pd.read_csv(p); raw[key] = df
    rows.append({"model":MODELS[key], "rows":len(df),
                 "labeled":int(df["label"].notna().sum()),
                 "parse_error":int((df["label"]=="parse_error").sum()),
                 "neutral_rate":f"{(df['label']=='neutral').mean():.1%}"})
health = pd.DataFrame(rows).set_index("model")
print(health.to_string())
for key, df in raw.items():
    assert df["label"].notna().sum() == len(df), f"{key}: incomplete"
    assert not (df["label"]=="parse_error").any(), f"{key}: has parse_error rows"
print("\nAll files complete, 0 parse_errors.")

# ---- merge integrity + wide pivot ----
ref = raw[loaded[0]][["turn_uid","text"]].drop_duplicates("turn_uid").set_index("turn_uid")
for key in loaded:
    assert set(raw[key]["turn_uid"]) == set(ref.index), f"{key}: turn_uid set differs"

base_cols   = ["turn_uid","bank","date","doc_type","speaker","speaker_role","turn_idx","text"]
present     = [c for c in base_cols if c in raw[loaded[0]].columns]
turns_wide  = raw[loaded[0]][present].copy()
for key in loaded:
    sub = raw[key][["turn_uid","label"]].rename(columns={"label":f"label_{key}"})
    turns_wide = turns_wide.merge(sub, on="turn_uid", how="inner")
    turns_wide[f"stanced_{key}"] = turns_wide[f"label_{key}"].isin(STANCED).astype(int)

n_models = len(loaded)
turns_wide["n_stanced"]  = turns_wide[[f"stanced_{k}" for k in loaded]].sum(axis=1)
turns_wide["split"]      = turns_wide["n_stanced"].between(1, n_models-1).astype(int)
turns_wide["all_neutral"]= (turns_wide["n_stanced"]==0).astype(int)
turns_wide["all_stanced"]= (turns_wide["n_stanced"]==n_models).astype(int)

# 3-way signed dispersion (the VAR-relevant disagreement target)
S3 = np.stack([turns_wide[f"label_{k}"].map(SCORE3).values for k in loaded], axis=1)
turns_wide["score_std3"]   = S3.std(axis=1, ddof=0)
turns_wide["mean_stance"]  = S3.mean(axis=1)
turns_wide["sign_conflict"]= ((S3==-1).any(1) & (S3==1).any(1)).astype(int)

def _ent(row):
    counts = [ [row[f"label_{k}"] for k in loaded].count(l) for l in LABEL_ORDER ]
    return entropy(counts, base=5)
turns_wide["label_entropy"] = turns_wide.apply(_ent, axis=1)

# doc_id for grouped split
fk = loaded[0]
if "doc_id" in raw[fk].columns:
    turns_wide = turns_wide.merge(raw[fk][["turn_uid","doc_id"]].drop_duplicates("turn_uid"),
                                  on="turn_uid", how="left")
else:
    turns_wide["doc_id"] = turns_wide["turn_uid"].astype(str).str.replace(r"_[0-9]+$","",regex=True)
if turns_wide["doc_id"].isna().any():
    fb = turns_wide["bank"].astype(str)+"_"+turns_wide["date"].astype(str)
    turns_wide["doc_id"] = turns_wide["doc_id"].fillna(fb)

turns_wide["word_count"] = turns_wide["text"].astype(str).str.split().str.len()
turns_grouped = turns_wide  # alias for the meeting-level cell

print(f"\nTurns: {len(turns_wide)}   split: {turns_wide['split'].mean():.1%}   "
      f"all-neutral: {turns_wide['all_neutral'].mean():.1%}   all-stanced: {turns_wide['all_stanced'].mean():.1%}")
print(f"score_std3 mean={turns_wide['score_std3'].mean():.3f}   sign_conflict={turns_wide['sign_conflict'].mean():.1%}")

In [ ]:
groups = turns_wide["doc_id"].astype(str).values
gss_o = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tv_idx, te_idx = next(gss_o.split(turns_wide, groups=groups))
trainval_df = turns_wide.iloc[tv_idx].reset_index(drop=True)
test_df_g   = turns_wide.iloc[te_idx].reset_index(drop=True)

gss_i = GroupShuffleSplit(n_splits=1, test_size=0.17647, random_state=SEED)
tr_rel, va_rel = next(gss_i.split(trainval_df, groups=trainval_df["doc_id"].astype(str).values))
train_df_g = trainval_df.iloc[tr_rel].reset_index(drop=True)
val_df_g   = trainval_df.iloc[va_rel].reset_index(drop=True)

trd, vad, ted = (train_df_g["doc_id"].astype(str), val_df_g["doc_id"].astype(str), test_df_g["doc_id"].astype(str))
ov = lambda a,b: len(set(a)&set(b))
assert ov(trd,vad)==0 and ov(trd,ted)==0 and ov(vad,ted)==0, "meeting leak!"
print(f"Grouped split  rows train={len(train_df_g)} val={len(val_df_g)} test={len(test_df_g)}")
print(f"               docs train={trd.nunique()} val={vad.nunique()} test={ted.nunique()}  (no overlap)")

# boolean masks aligned to turns_wide order
tr_uid, va_uid, te_uid = set(train_df_g.turn_uid), set(val_df_g.turn_uid), set(test_df_g.turn_uid)
tr_m = turns_wide.turn_uid.isin(tr_uid).values
va_m = turns_wide.turn_uid.isin(va_uid).values
te_m = turns_wide.turn_uid.isin(te_uid).values
y_tr = train_df_g["split"].values
y_te = test_df_g["split"].values

In [ ]:
def cohens_kappa(left, right):
    paired = pd.DataFrame({"l":left,"r":right}).dropna()
    if paired.empty: return float("nan")
    agree = (paired["l"]==paired["r"]).mean()
    conf  = pd.crosstab(paired["l"], paired["r"]); tot = conf.to_numpy().sum()
    exp   = (conf.sum(1)/tot).mul(conf.sum(0)/tot, fill_value=0).sum()
    return float("nan") if exp==1 else (agree-exp)/(1-exp)

def fleiss_kappa(df, cols, cats):
    n, r = len(df), len(cols)
    counts = pd.DataFrame({c:(df[cols]==c).sum(1) for c in cats})
    p_j   = counts.sum()/(n*r)
    P_i   = ((counts**2).sum(1)-r)/(r*(r-1))
    P_e   = (p_j**2).sum()
    return float("nan") if P_e==1 else (P_i.mean()-P_e)/(1-P_e)

label_cols = [f"label_{k}" for k in loaded]

# corpus structure
print("Corpus structure (all turns):")
print(f"  all-neutral {turns_wide['all_neutral'].mean():.1%} | "
      f"split {turns_wide['split'].mean():.1%} | all-stanced {turns_wide['all_stanced'].mean():.1%}")
print(f"  sign-conflict (a dove AND a hawk on same turn): {turns_wide['sign_conflict'].mean():.1%} "
      f"({turns_wide.loc[turns_wide['split'].astype(bool),'sign_conflict'].mean():.1%} of split turns)")

# hierarchical kappa: stancedness vs direction
stanced_cols = []
for k in loaded:
    c = f"st_{k}"; turns_wide[c] = np.where(turns_wide[f"label_{k}"].isin(STANCED), "stanced", "neutral"); stanced_cols.append(c)
k_stanced = fleiss_kappa(turns_wide, stanced_cols, ["neutral","stanced"])

dir_cols = []
for k in loaded:
    c = f"dir_{k}"; turns_wide[c] = turns_wide[f"label_{k}"].map(DIR2); dir_cols.append(c)
# pairwise direction kappa among turns where both raters are directional
dir_ks = []
for a,b in combinations(dir_cols,2):
    sub = turns_wide[[a,b]].dropna()
    if len(sub) > 30: dir_ks.append(cohens_kappa(sub[a], sub[b]))
k_direction = float(np.nanmean(dir_ks))

print(f"\nHierarchical agreement:")
print(f"  kappa_stanced  (neutral vs stanced, Fleiss, 6 raters): {k_stanced:.3f}")
print(f"  kappa_direction(dove vs hawk | both stanced, mean pairwise): {k_direction:.3f}")
print("  -> disagreement is about WHETHER a turn is stanced, not WHICH way.")

print("\nNeutral-rate (leniency) ordering:")
for k in sorted(loaded, key=lambda k:-(turns_wide[f'label_{k}']=='neutral').mean()):
    print(f"  {MODELS[k]:18s} {(turns_wide[f'label_{k}']=='neutral').mean():.1%}")

In [ ]:
# ---- Baseline 1: dictionary ----
HAWK_WORDS = {"hike","hikes","hiking","tighten","tightening","tightened","raise","raises","raising",
    "restrictive","restriction","inflation","inflationary","overshoot","overheating","hawkish",
    "normalisation","normalization","unwind"}
DOVE_WORDS = {"cut","cuts","cutting","accommodation","accommodative","stimulus","easing","ease",
    "support","lower","lowering","dovish","expansionary","unconventional","qe","purchase","below","undershoot"}
def dict_score(t):
    toks = re.findall(r"[a-z]+", str(t).lower())
    return sum(w in HAWK_WORDS for w in toks) - sum(w in DOVE_WORDS for w in toks)

Xd_tr = train_df_g["text"].apply(dict_score).values.reshape(-1,1)
Xd_te = test_df_g["text"].apply(dict_score).values.reshape(-1,1)
auc_dict = roc_auc_score(y_te, LogisticRegression(class_weight="balanced",max_iter=1000)
                                .fit(Xd_tr,y_tr).predict_proba(Xd_te)[:,1])

# ---- Baseline 2: TF-IDF (split AUC + score_std3 Spearman) ----
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3, sublinear_tf=True)
Xt_tr = tfidf.fit_transform(train_df_g["text"]); Xt_te = tfidf.transform(test_df_g["text"])
lr_tfidf = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0).fit(Xt_tr, y_tr)
auc_tfidf = roc_auc_score(y_te, lr_tfidf.predict_proba(Xt_te)[:,1])

rg_tfidf = Ridge(alpha=1.0).fit(Xt_tr, train_df_g["score_std3"].values)
rho_tfidf = spearmanr(test_df_g["score_std3"].values, rg_tfidf.predict(Xt_te)).correlation

print(f"Dictionary  split-AUC : {auc_dict:.3f}")
print(f"TF-IDF      split-AUC : {auc_tfidf:.3f}   score_std3 Spearman: {rho_tfidf:.3f}")

# ---- SHAP top words driving split ----
try:
    import shap
    expl = shap.LinearExplainer(lr_tfidf, Xt_tr, feature_perturbation="interventional")
    sv = expl.shap_values(Xt_te); feats = tfidf.get_feature_names_out()
    top = np.argsort(np.abs(sv).mean(0))[::-1][:25]
    plt.figure(figsize=(9,6))
    plt.barh(range(25), np.abs(sv).mean(0)[top][::-1], color="#4C72B0")
    plt.yticks(range(25), [feats[i] for i in top[::-1]], fontsize=8)
    plt.xlabel("mean |SHAP|"); plt.title("TF-IDF words predicting LLM disagreement (split)")
    plt.tight_layout(); plt.savefig("tfidf_shap_split.png", dpi=130, bbox_inches="tight"); plt.show()
except Exception as e:
    print(f"(SHAP skipped: {e})")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DisagreeDataset(Dataset):
    def __init__(self, df, max_length=512):
        self.df = df.reset_index(drop=True); self.max_length = max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = tokenizer(str(row["text"]), max_length=self.max_length, padding="max_length",
                        truncation=True, return_tensors="pt")
        return {"input_ids":enc["input_ids"].squeeze(0),
                "attention_mask":enc["attention_mask"].squeeze(0),
                "split":torch.tensor(float(row["split"]), dtype=torch.float32),
                "std":torch.tensor(float(row["score_std3"]), dtype=torch.float32)}

class DisagreeModel(nn.Module):
    def __init__(self, encoder_name):
        super().__init__()
        self.encoder    = DebertaV2Model.from_pretrained(encoder_name)
        self.split_head = nn.Linear(768, 1)
        self.std_head   = nn.Linear(768, 1)
    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0,:].float()
        return self.split_head(cls).squeeze(-1), self.std_head(cls).squeeze(-1), cls

model = DisagreeModel(MODEL_NAME).to(device)
BATCH = 8
train_loader = DataLoader(DisagreeDataset(train_df_g), batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(DisagreeDataset(val_df_g),   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f"Encoder params: {sum(p.numel() for p in model.encoder.parameters()):,}")

In [ ]:
SKIP_TRAINING = "auto"   # "auto" = load checkpoint if present else train
CKPT_NAME = "model_turns6_disagree.pt"
if os.path.exists("/content"):
    try:
        from google.colab import drive; drive.mount("/drive", force_remount=False)
        CKPT_DIR = "/drive/MyDrive/central_bank_spillovers/multihead_turns"
        os.makedirs(CKPT_DIR, exist_ok=True); CKPT_PATH = f"{CKPT_DIR}/{CKPT_NAME}"
    except Exception:
        CKPT_PATH = CKPT_NAME
else:
    CKPT_PATH = CKPT_NAME

from transformers import get_linear_schedule_with_warmup

p_split = float(np.mean(y_tr))
pos_weight = torch.tensor([(1-p_split)/max(p_split,1e-6)], device=device)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
mse = nn.MSELoss()
STD_W   = 0.5    # weight on the regression loss
ENC_LR  = 1e-5   # DeBERTa-v3 is unstable at 2e-5 without warmup

def train_model(epochs=3, grad_acc=4):
    for epoch in range(1, epochs+1):
        full = epoch > 1
        for p in model.encoder.parameters(): p.requires_grad = full
        opt = torch.optim.AdamW([
            {"params":model.encoder.parameters(),    "lr":(ENC_LR if full else 0.0), "weight_decay":0.01},
            {"params":model.split_head.parameters(), "lr":1e-3, "weight_decay":0.01},
            {"params":model.std_head.parameters(),   "lr":1e-3, "weight_decay":0.01}])
        n_steps = max(1, len(train_loader)//grad_acc)
        sched   = get_linear_schedule_with_warmup(opt, int(0.1*n_steps), n_steps)  # 10% linear warmup
        model.train(); opt.zero_grad(); run=0.0; skipped=0
        for step,b in enumerate(train_loader,1):
            ids=b["input_ids"].to(device); mask=b["attention_mask"].to(device)
            sl, sd, _ = model(ids, mask)
            loss = (bce(sl, b["split"].to(device)) + STD_W*mse(sd, b["std"].to(device)))/grad_acc
            if not torch.isfinite(loss):          # guard: skip a non-finite step instead of poisoning weights
                skipped+=1; opt.zero_grad(); continue
            loss.backward(); run += loss.item()*grad_acc
            if step % grad_acc == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sched.step(); opt.zero_grad()
            if step % 50 == 0: print(f"  epoch {epoch} step {step}/{len(train_loader)} loss={run/step:.4f}")
        print(f"epoch {epoch} done ({'full FT' if full else 'frozen'})  skipped_nan_steps={skipped}")

@torch.no_grad()
def model_is_finite():
    model.eval()
    b = next(iter(val_loader))
    sl, sd, _ = model(b["input_ids"].to(device), b["attention_mask"].to(device))
    return bool(torch.isfinite(sl).all() and torch.isfinite(sd).all())

loaded_ok = False
if SKIP_TRAINING in (True,"auto") and os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ck["model_state_dict"]); model = model.float().to(device)
    if model_is_finite():
        loaded_ok = True; print(f"Loaded checkpoint: {CKPT_PATH}")
    else:
        print(f"Checkpoint {CKPT_PATH} produces NaN — ignoring and retraining. Delete it to be safe.")
if not loaded_ok:
    print("Training disagreement model from scratch ...")
    train_model()
    if model_is_finite():
        try: torch.save({"model_state_dict":model.state_dict()}, CKPT_PATH); print(f"Saved: {CKPT_PATH}")
        except Exception as e: print(f"(no save: {e})")
    else:
        print("WARNING: model still non-finite after training — NOT saving. Lower ENC_LR / STD_W and re-run.")

In [ ]:
@torch.no_grad()
def predict_all(m, df, batch_size=16):
    m.eval(); ld = DataLoader(DisagreeDataset(df), batch_size=batch_size, shuffle=False, num_workers=2)
    sp, st, cl = [], [], []
    for b in ld:
        ids=b["input_ids"].to(device); mask=b["attention_mask"].to(device)
        sl, sd, cls = m(ids, mask)
        sp.append(torch.sigmoid(sl).cpu().numpy()); st.append(sd.cpu().numpy()); cl.append(cls.cpu().numpy())
    return np.concatenate(sp), np.concatenate(st), np.vstack(cl)

@torch.no_grad()
def extract_cls(encoder, df, batch_size=16):
    encoder.eval(); ld = DataLoader(DisagreeDataset(df), batch_size=batch_size, shuffle=False, num_workers=2)
    out=[]
    for b in ld:
        ids=b["input_ids"].to(device); mask=b["attention_mask"].to(device)
        out.append(encoder(input_ids=ids, attention_mask=mask).last_hidden_state[:,0,:].float().cpu().numpy())
    return np.vstack(out)

# fine-tuned predictions over ALL turns (aligned to turns_wide order)
ft_split, ft_std, X_all_ft = predict_all(model, turns_wide)
assert np.isfinite(ft_split).all() and np.isfinite(ft_std).all(), (
    "Non-finite predictions: the fine-tuned model diverged. Re-run the training cell "
    "(it now uses warmup + a NaN guard) on a FRESH model, and delete any saved NaN checkpoint.")
turns_wide["pred_split"] = ft_split
turns_wide["pred_std3"]  = ft_std

auc_ft  = roc_auc_score(y_te, ft_split[te_m])
rho_ft  = spearmanr(turns_wide.loc[te_m,"score_std3"], ft_std[te_m]).correlation

# frozen pretrained baseline
frozen = DebertaV2Model.from_pretrained(MODEL_NAME).to(device).float()
X_fr = extract_cls(frozen, turns_wide)
sc_b = StandardScaler().fit(X_fr[tr_m])
auc_fr = roc_auc_score(y_te, LogisticRegression(max_iter=1000,class_weight="balanced")
                              .fit(sc_b.transform(X_fr[tr_m]), y_tr).predict_proba(sc_b.transform(X_fr[te_m]))[:,1])
rho_fr = spearmanr(turns_wide.loc[te_m,"score_std3"],
                   Ridge(alpha=1.0).fit(sc_b.transform(X_fr[tr_m]), turns_wide.loc[tr_m,"score_std3"])
                                   .predict(sc_b.transform(X_fr[te_m]))).correlation
del frozen; import gc; gc.collect(); torch.cuda.empty_cache()

ladder = pd.DataFrame([
    {"method":"Dictionary",         "split_AUC":auc_dict, "std3_Spearman":np.nan},
    {"method":"TF-IDF",             "split_AUC":auc_tfidf, "std3_Spearman":rho_tfidf},
    {"method":"Frozen DeBERTa",     "split_AUC":auc_fr,    "std3_Spearman":rho_fr},
    {"method":"Fine-tuned DeBERTa", "split_AUC":auc_ft,    "std3_Spearman":rho_ft},
])
print("=== Disagreement-prediction ladder (grouped split) ===")
print(ladder.round(3).to_string(index=False))
print(f"\nFine-tuning lift over TF-IDF:  split-AUC {auc_ft-auc_tfidf:+.3f}   std3-Spearman {rho_ft-rho_tfidf:+.3f}")
ladder.to_csv("probe_auc_comparison.csv", index=False)

In [ ]:
# Train a stancedness-boundary classifier on UNANIMOUS turns only (consensus pseudo-labels):
#   all-neutral (0)  vs  all-stanced (1).   Then ask where the SPLIT turns fall.
unanim = turns_wide[(turns_wide["all_neutral"]==1)|(turns_wide["all_stanced"]==1)].copy()
unanim["y_bnd"] = unanim["all_stanced"].values
u_tr = unanim[unanim["turn_uid"].isin(tr_uid)]
print(f"Unanimous training turns: {len(u_tr)}  (neutral {int((u_tr.y_bnd==0).sum())} / stanced {int((u_tr.y_bnd==1).sum())})")

# --- primary: interpretable TF-IDF boundary model ---
bnd_tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3, sublinear_tf=True)
Xb_tr = bnd_tfidf.fit_transform(u_tr["text"])
bnd_clf = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced").fit(Xb_tr, u_tr["y_bnd"].values)
turns_wide["P_stanced"] = bnd_clf.predict_proba(bnd_tfidf.transform(turns_wide["text"]))[:,1]

# --- secondary: embedding-space boundary margin (not purely lexical) ---
u_tr_mask = turns_wide["turn_uid"].isin(set(u_tr["turn_uid"])).values
sc_e = StandardScaler().fit(X_all_ft[u_tr_mask])
emb_clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(
    sc_e.transform(X_all_ft[u_tr_mask]), turns_wide.loc[u_tr_mask,"all_stanced"].values)
turns_wide["P_stanced_emb"] = emb_clf.predict_proba(sc_e.transform(X_all_ft))[:,1]

def grp(row): return "all_neutral" if row.all_neutral else ("all_stanced" if row.all_stanced else "split")
turns_wide["grp"] = turns_wide.apply(grp, axis=1)

print("\n=== Boundary-case test: is P_stanced near 0.5 for SPLIT turns? ===")
for col in ["P_stanced","P_stanced_emb"]:
    print(f"\n[{col}]")
    for g in ["all_neutral","split","all_stanced"]:
        s = turns_wide.loc[turns_wide.grp==g, col]
        print(f"  {g:12s} n={len(s):4d}  meanP={s.mean():.3f}  mean|P-0.5|={ (s-0.5).abs().mean():.3f}  "
              f"share[0.4,0.6]={s.between(0.4,0.6).mean():.1%}")

# plot
fig, axes = plt.subplots(1,2, figsize=(13,4.5))
colors = {"all_neutral":"#4C72B0","split":"#C44E52","all_stanced":"#55A868"}
for ax,col,ttl in zip(axes, ["P_stanced","P_stanced_emb"], ["TF-IDF boundary model","Fine-tuned embedding"]):
    for g in ["all_neutral","split","all_stanced"]:
        ax.hist(turns_wide.loc[turns_wide.grp==g,col], bins=30, alpha=0.5, density=True, label=g, color=colors[g])
    ax.axvline(0.5, color="k", ls="--", lw=0.8); ax.set_title(ttl); ax.set_xlabel("P(stanced)")
axes[0].legend(); plt.suptitle("Split turns are stancedness-boundary cases", y=1.02)
plt.tight_layout(); plt.savefig("boundary_distribution.png", dpi=130, bbox_inches="tight"); plt.show()

# --- length control: boundary-ness is not just length ---
turns_wide["len_tercile"] = pd.qcut(turns_wide["word_count"], 3, labels=["short","mid","long"])
print("\n=== Length control: mean |P_stanced-0.5| by length tercile ===")
ctrl = (turns_wide.assign(absm=(turns_wide["P_stanced"]-0.5).abs())
        .groupby(["len_tercile", turns_wide.grp.eq("split").map({True:"split",False:"unanimous"})], observed=True)["absm"]
        .mean().unstack())
print(ctrl.round(3).to_string())
print("-> split turns sit closer to the boundary within every length tercile.")

In [ ]:
# Cue families (regex on TF-IDF feature strings); reused/extended from the lab notebook.
CUE_FAMILIES = {
  "condition_inflation": [r"inflation", r"price", r"undershoot", r"overshoot", r"disinflation"],
  "condition_labour_growth": [r"labou?r", r"employ", r"unemploy", r"wage", r"growth", r"demand", r"activity", r"output"],
  "policy_action": [r"rate", r"hike", r"raise", r"cut", r"tighten", r"easing", r"restrictive", r"accommodat", r"normalis", r"normaliz", r"stimulus"],
  "hedging": [r"may\b", r"might", r"could", r"would", r"likely", r"somewhat", r"broadly", r"gradual", r"moderate"],
  "conditionality_guidance": [r"\bif\b", r"depend", r"data[- ]dependent", r"conditional", r"as appropriate", r"stand ready", r"monitor", r"incoming"],
  "mixed_two_sided_risk": [r"\brisk", r"uncertain", r"two[- ]sided", r"balance", r"on the other hand", r"however", r"shock", r"volatil"],
}
feats = np.array(bnd_tfidf.get_feature_names_out())
coef  = bnd_clf.coef_[0]   # + => pushes toward STANCED, - => toward NEUTRAL
rows = []
for fam, pats in CUE_FAMILIES.items():
    m = np.array([any(re.search(p, f) for p in pats) for f in feats])
    if m.sum()==0: continue
    rows.append({"family":fam, "n_features":int(m.sum()),
                 "mean_coef":coef[m].mean(), "mean_abs_coef":np.abs(coef[m]).mean(),
                 "sum_abs_coef":np.abs(coef[m]).sum()})
fam_tbl = pd.DataFrame(rows).sort_values("sum_abs_coef", ascending=False)
print("=== Cue families in the boundary model (sign: + stanced / - neutral) ===")
print(fam_tbl.round(3).to_string(index=False))
fam_tbl.to_csv("boundary_cue_families.csv", index=False)

# ---- Sentence leave-one-out on the most-boundary split turns ----
def fam_of(text):
    hits = [fam for fam,pats in CUE_FAMILIES.items() if any(re.search(p, str(text), re.I) for p in pats)]
    return ", ".join(hits) if hits else "(none)"

split_turns = turns_wide[turns_wide.grp=="split"].copy()
split_turns["absm"] = (split_turns["P_stanced"]-0.5).abs()
topN = split_turns.sort_values("absm").head(40)   # most boundary-ambiguous

loo_rows = []
for _, row in topN.iterrows():
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(row["text"])) if len(s.split())>=4]
    if len(sents) < 2: continue
    full_p = row["P_stanced"]; full_absm = abs(full_p-0.5)
    best = None
    for i in range(len(sents)):
        masked = " ".join(sents[:i]+sents[i+1:])
        p = bnd_clf.predict_proba(bnd_tfidf.transform([masked]))[:,1][0]
        d = abs(p-0.5) - full_absm   # >0: removing this sentence makes the turn more decisive
        if best is None or d > best[0]: best = (d, sents[i], p)
    if best and best[0] > 0:
        loo_rows.append({"bank":row.get("bank",""), "date":row.get("date",""),
                         "full_P":round(full_p,3), "P_without":round(best[2],3),
                         "decisiveness_gain":round(best[0],3),
                         "family":fam_of(best[1]), "ambiguity_sentence":best[1][:160]})
loo = pd.DataFrame(loo_rows).sort_values("decisiveness_gain", ascending=False)
print(f"\n=== Sentence leave-one-out: language that creates boundary ambiguity (top {len(loo)} split turns) ===")
print(loo.head(15).to_string(index=False))
print("\nCue-family frequency among ambiguity-creating sentences:")
print(loo["family"].value_counts().to_string())
loo.to_csv("boundary_sentence_loo.csv", index=False)

In [ ]:
df = turns_wide.copy()
df["date_dt"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
na = df["date_dt"].isna()
df.loc[na,"date_dt"] = pd.to_datetime(df.loc[na,"date"].astype(str), format="%Y%m", errors="coerce")
df["bnd_uncertainty"] = 1 - 2*(df["P_stanced"]-0.5).abs()   # 1 at boundary, 0 at extremes

# per-model meeting-mean stance -> model-choice spread
per = pd.DataFrame({k: df.assign(_s=df[f"label_{k}"].map(SCORE3)).groupby(["bank","date_dt"])["_s"].mean()
                    for k in loaded})
spread = per.std(axis=1, ddof=0).rename("model_choice_spread").reset_index()

is_test = df["turn_uid"].isin(te_uid).astype(int)
meet = (df.assign(is_test=is_test)
          .groupby(["bank","date_dt"])
          .agg(mean_stance=("mean_stance","mean"), disagreement=("score_std3","mean"),
               disagreement_pred=("pred_std3","mean"), bnd_uncertainty=("bnd_uncertainty","mean"),
               sign_conflict_rate=("sign_conflict","mean"), n_turns=("turn_uid","size"),
               is_test=("is_test","max")).reset_index()
          .merge(spread, on=["bank","date_dt"], how="left").sort_values(["bank","date_dt"]))

mt = meet[meet.is_test==1]
print(f"Turn-level test Spearman (pred vs actual disagreement): "
      f"{spearmanr(df.loc[te_m,'score_std3'], df.loc[te_m,'pred_std3']).correlation:.3f}")
print(f"Meeting-level test Spearman (n={len(mt)} held-out meetings): "
      f"{spearmanr(mt['disagreement'], mt['disagreement_pred']).correlation:.3f}")
print(f"Corr(disagreement, model_choice_spread), all meetings: "
      f"{spearmanr(meet['disagreement'], meet['model_choice_spread']).correlation:.3f}  <- RQ2")

banks = sorted(meet.bank.unique())
fig, axes = plt.subplots(len(banks),1, figsize=(14,3.2*len(banks)), sharex=True)
if len(banks)==1: axes=[axes]
for ax,bk in zip(axes,banks):
    b = meet[meet.bank==bk]
    ax.plot(b.date_dt, b.disagreement, color="#C44E52", lw=1.6, label="actual disagreement")
    ax.plot(b.date_dt, b.disagreement_pred, color="#4C72B0", lw=1.3, alpha=0.85, label="text-predicted")
    ax.set_title(bk); ax.set_ylabel("mean score_std3")
axes[0].legend(ncol=2, fontsize=8)
plt.suptitle("Meeting-level LLM disagreement: actual vs text-predicted", y=1.01)
plt.xlabel("Date"); plt.tight_layout()
plt.savefig("meeting_disagreement_timeline.png", dpi=130, bbox_inches="tight"); plt.show()

meet.to_csv("meeting_disagreement_for_var.csv", index=False)
print("\nSaved meeting_disagreement_for_var.csv (one row per bank-meeting)")
print(meet.head().round(3).to_string(index=False))

In [ ]:
import shutil
outputs = ["probe_auc_comparison.csv","tfidf_shap_split.png","boundary_distribution.png",
           "boundary_cue_families.csv","boundary_sentence_loo.csv",
           "meeting_disagreement_timeline.png","meeting_disagreement_for_var.csv"]
if os.path.exists("/content"):
    try:
        if not os.path.exists("/drive/MyDrive"):
            from google.colab import drive; drive.mount("/drive", force_remount=False)
        DD = "/drive/MyDrive/central_bank_spillovers/multihead_turns"; os.makedirs(DD, exist_ok=True)
        for f in outputs:
            if os.path.exists(f): shutil.copy(f, os.path.join(DD,f)); print(f"  saved: {f}")
    except Exception as e: print(f"(drive copy skipped: {e})")
    from google.colab import files
    for f in outputs:
        if os.path.exists(f): files.download(f)
else:
    for f in outputs:
        print(f"  {'OK  ' if os.path.exists(f) else 'MISS'} {f}")